# House Prices — Step 3: One-Hot Encoding

One-Hot Encoding creates a new binary (0/1) column for each category. Unlike Label Encoding, it doesn't imply any order between categories, which makes it the right choice for **nominal** columns (no natural ranking) — especially ones with **low cardinality** (few unique values).

The trade-off: every extra category adds an extra column. For a high-cardinality column like `Neighborhood` (25 unique values), one-hot encoding alone would add 24 new columns just for that one feature. This notebook demonstrates the technique, shows that dimensionality cost directly, and then applies it across the dataset.

## 1. Load the cleaned dataset

Continuing from `train_cleaned.csv` (Step 1).

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 15)

df = pd.read_csv('train_cleaned.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,...,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,...,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,...,NaN,0,9,2008,WD,Normal,223500


## 2. Check cardinality of categorical columns

Before one-hot encoding anything, it's worth checking how many unique categories each column has — this tells us which columns are cheap to encode and which will blow up dimensionality.

In [2]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
cardinality = df[cat_cols].nunique().sort_values(ascending=False)
print(f"Total categorical columns: {len(cat_cols)}")
cardinality

Total categorical columns: 43


/tmp/ipykernel_91/592919064.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


Neighborhood     25
Exterior2nd      16
Exterior1st      15
Condition1        9
SaleType          9
HouseStyle        8
RoofMatl          8
Condition2        8
Functional        7
BsmtFinType2      6
RoofStyle         6
BsmtFinType1      6
SaleCondition     6
Heating           6
Foundation        6
GarageType        6
ExterCond         5
LotConfig         5
MSZoning          5
GarageCond        5
GarageQual        5
HeatingQC         5
Electrical        5
BldgType          5
FireplaceQu       5
LandContour       4
LotShape          4
KitchenQual       4
MiscFeature       4
Fence             4
BsmtCond          4
ExterQual         4
BsmtExposure      4
BsmtQual          4
LandSlope         3
PoolQC            3
GarageFinish      3
PavedDrive        3
MasVnrType        3
Utilities         2
Alley             2
Street            2
CentralAir        2
dtype: int64

## 3. Low-cardinality example: `Street`

`Street` only has 2 categories (`Pave`, `Grvl`) — a textbook case for one-hot encoding. We use `pd.get_dummies` with `drop_first=True`, which drops one category to avoid the "dummy variable trap" (perfect multicollinearity, since if you know all-but-one dummy value, the last one is fully determined).

In [3]:
street_dummies = pd.get_dummies(df['Street'], prefix='Street', drop_first=True)
pd.concat([df[['Street']], street_dummies], axis=1).drop_duplicates()

,Street,Street_Pave
0,Pave,True
52,Grvl,False


## 4. Moderate-cardinality example: `MSZoning`

`MSZoning` has 5 categories. One-hot encoding turns it into 4 binary columns (with `drop_first=True`).

In [4]:
zoning_dummies = pd.get_dummies(df['MSZoning'], prefix='MSZoning', drop_first=True)
pd.concat([df[['MSZoning']], zoning_dummies], axis=1).drop_duplicates()

,MSZoning,MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM
0,RL,False,False,True,False
8,RM,False,False,False,True
30,C (all),False,False,False,False
47,FV,True,False,False,False
341,RH,False,True,False,False


## 5. High-cardinality warning: `Neighborhood`

`Neighborhood` has 25 categories. One-hot encoding it alone adds 24 new columns. This isn't wrong, but it's worth seeing the actual cost before doing it dataset-wide — for very high-cardinality columns, Target Encoding (covered in the next notebook) is often a more compact and effective alternative.

In [5]:
neighborhood_dummies = pd.get_dummies(df['Neighborhood'], prefix='Neighborhood', drop_first=True)
print("Original columns: 1  ->  One-hot columns:", neighborhood_dummies.shape[1])
neighborhood_dummies.head(3)

Original columns: 1  ->  One-hot columns: 24


,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,...,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True
2,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False


## 6. Apply One-Hot Encoding to all categorical columns

We now one-hot encode every remaining categorical column with `pd.get_dummies(..., drop_first=True)`, producing a fully numeric dataframe. As calculated above, this adds roughly 200 extra columns to the dataset — worth keeping in mind for model training time and memory.

In [6]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

df_onehot = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("Original shape:", df.shape)
print("One-hot encoded shape:", df_onehot.shape)
print("Columns added:", df_onehot.shape[1] - df.shape[1])

Original shape: (1460, 81)
One-hot encoded shape: (1460, 246)
Columns added: 165


/tmp/ipykernel_91/1390342580.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


## 7. Save the one-hot encoded dataset

Saved separately so we can compare against the Label Encoding and Target Encoding versions.

In [7]:
df_onehot.to_csv('train_onehot_encoded.csv', index=False)
print("Saved train_onehot_encoded.csv with shape:", df_onehot.shape)

Saved train_onehot_encoded.csv with shape: (1460, 246)
